# 基于PCA特征的KMeans聚类分析

本notebook基于PCA特征进行KMeans聚类，并为每个class_name绘制散点图


In [1]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100


In [3]:
# 读取PCA特征数据
pca_features = pd.read_csv("pca_features.csv")



In [4]:
# 提取PCA特征列（PC1-PC50）
pca_columns = [col for col in pca_features.columns if col.startswith('PC')]
X_pca = pca_features[pca_columns].values
print(f"PCA特征维度: {X_pca.shape}")
print(f"PCA特征列: {pca_columns[:10]}... (共{len(pca_columns)}列)")


PCA特征维度: (22248, 50)
PCA特征列: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']... (共50列)


In [6]:
n_clusters = 20

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, max_iter=300)
cluster_labels = kmeans.fit_predict(X_pca)

# 将聚类结果添加到数据框
pca_features['kmeans_cluster'] = cluster_labels



In [7]:
def generate_all_clusters_scatter_pdf():
    """
    生成包含所有cluster的散点图PDF，并在每个cluster位置标注cluster名称
    """
    # 创建PDF文件
    output_path = "pca_all_clusters_scatter_plot.pdf"
    
    # 获取所有唯一的cluster
    unique_clusters = sorted(pca_features['kmeans_cluster'].unique())
    print(f"找到 {len(unique_clusters)} 个不同的cluster")
    
    # 创建图形
    fig, ax = plt.subplots(figsize=(16, 16))
    
    # 为每个cluster分配不同的颜色
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters)))
    
    # 计算每个cluster的中心位置用于标注
    cluster_centers = {}
    
    # 绘制每个cluster
    for i, cluster_id in enumerate(unique_clusters):
        cluster_mask = pca_features['kmeans_cluster'] == cluster_id
        
        if cluster_mask.sum() > 0:
            # 绘制cluster点
            ax.scatter(pca_features.loc[cluster_mask, 'PC1'],
                      pca_features.loc[cluster_mask, 'PC2'],
                      c=[colors[i]], alpha=0.7, s=20, 
                      label=f'Cluster {cluster_id} ({cluster_mask.sum():,} points)')
            
            # 计算cluster中心位置
            center_x = pca_features.loc[cluster_mask, 'PC1'].mean()
            center_y = pca_features.loc[cluster_mask, 'PC2'].mean()
            cluster_centers[cluster_id] = (center_x, center_y)
    
    # 在cluster中心位置添加标注
    for cluster_id, (center_x, center_y) in cluster_centers.items():
        ax.annotate(f'C{cluster_id}', 
                   (center_x, center_y),
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=20, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                   ha='left')
    
    ax.set_xlabel('t-SNE 1', fontsize=18)
    ax.set_ylabel('t-SNE 2', fontsize=18)
    ax.grid(False)
    
    plt.tight_layout()
    
    with PdfPages(output_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight', dpi=300)
    
    plt.close(fig)
    
    print(f"\nPDF文件已生成: {output_path}")
    return output_path

generate_all_clusters_scatter_pdf()


找到 20 个不同的cluster

PDF文件已生成: pca_all_clusters_scatter_plot.pdf


'pca_all_clusters_scatter_plot.pdf'

In [18]:
# 创建cluster和class_name的交叉表
cluster_class_crosstab = pd.crosstab(pca_features['kmeans_cluster'], pca_features['class_name'])
print(f"交叉表形状: {cluster_class_crosstab.shape}")

# 找出每个cluster的主要class（出现次数>=10的class）
cluster_summarize = cluster_class_crosstab.copy()
cluster_summarize[cluster_summarize < 10] = 0
cluster_summarize[cluster_summarize >= 10] = 1

cluster_label = {}
for i in range(len(cluster_summarize)):
    cluster_label[i] = list(cluster_summarize.columns[np.where(cluster_summarize.iloc[i, :] == 1)[0]])



交叉表形状: (20, 1854)


In [9]:
def generate_per_class_scatter_plots(max_classes=50):
    """
    为每个class_name生成单独的散点图，显示该class在所有聚类中的分布
    """
    output_path = "pca_per_class_scatter_plots.pdf"
    
    # 获取所有唯一的class_name
    unique_classes = sorted(pca_features['class_name'].unique())
    print(f"总共有 {len(unique_classes)} 个不同的class")
    
    # 限制绘制的class数量（如果太多会很慢）
    if len(unique_classes) > max_classes:
        print(f"只绘制前 {max_classes} 个class的散点图")
        unique_classes = unique_classes[:max_classes]
    
    # 创建PDF
    with PdfPages(output_path) as pdf:
        for class_name in tqdm(unique_classes, desc="绘制class散点图"):
            fig, ax = plt.subplots(figsize=(12, 10))
            
            # 获取该class的所有点
            class_mask = pca_features['class_name'] == class_name
            class_data = pca_features[class_mask]
            
            # 绘制背景点（其他所有点，灰色）
            other_mask = ~class_mask
            ax.scatter(pca_features.loc[other_mask, 'PC1'],
                      pca_features.loc[other_mask, 'PC2'],
                      c='lightgray', alpha=0.3, s=10, label='其他类别')
            
            # 绘制目标class的点，按cluster着色
            unique_clusters = sorted(class_data['kmeans_cluster'].unique())
            colors = plt.cm.tab20(np.linspace(0, 1, 20))  # 使用固定的20种颜色
            
            for cluster_id in unique_clusters:
                cluster_mask = class_data['kmeans_cluster'] == cluster_id
                cluster_class_data = class_data[cluster_mask]
                
                ax.scatter(cluster_class_data['PC1'],
                          cluster_class_data['PC2'],
                          c=[colors[cluster_id]], alpha=0.8, s=50,
                          label=f'Cluster {cluster_id} ({len(cluster_class_data)} points)',
                          edgecolors='black', linewidths=0.5)
            
            ax.set_xlabel('t-SNE 1', fontsize=14)
            ax.set_ylabel('t-SNE 2', fontsize=14)
            ax.set_title(f'Class: {class_name} (共{len(class_data)}个样本)', fontsize=16, fontweight='bold')
            ax.legend(loc='best', fontsize=10, markerscale=0.8)
            ax.grid(False)
            
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight', dpi=150)
            plt.close(fig)
    
    print(f"\n已生成per-class散点图PDF: {output_path}")
    return output_path

# 生成前50个class的散点图
generate_per_class_scatter_plots(max_classes=50)


总共有 1854 个不同的class
只绘制前 50 个class的散点图


绘制class散点图: 100%|██████████| 50/50 [00:12<00:00,  4.14it/s]


已生成per-class散点图PDF: pca_per_class_scatter_plots.pdf


'pca_per_class_scatter_plots.pdf'

In [11]:
from scipy.spatial.distance import cdist, pdist
from itertools import combinations

# tsne_features 已经包含了所有需要的数据（tSNE1, tSNE2, label, class_name）
data = pca_features.copy()

# 1. 计算同一个class_name内点之间的距离
intra_class_distances = []

print("计算类内距离...")
for class_name in tqdm(data['class_name'].unique()):
    # 获取同一类别的所有点
    class_data = data[data['class_name'] == class_name][['PC1', 'PC2']].values
    
    if len(class_data) > 1:
        # 计算该类别内所有点对之间的欧氏距离
        distances = pdist(class_data, metric='euclidean')
        intra_class_distances.extend(distances)

intra_class_distances = np.array(intra_class_distances)

print(f"类内距离数量: {len(intra_class_distances)}")
print(f"类内平均距离: {np.mean(intra_class_distances):.4f}")
print(f"类内距离标准差: {np.std(intra_class_distances):.4f}")
print(f"类内距离中位数: {np.median(intra_class_distances):.4f}")

计算类内距离...


100%|██████████| 1854/1854 [00:01<00:00, 1318.47it/s]


类内距离数量: 122364
类内平均距离: 0.1344
类内距离标准差: 0.0883
类内距离中位数: 0.1156


In [ ]:
n_random_samples = len(intra_class_distances)

print(f"\n计算随机点对距离（采样 {n_random_samples} 对）...")
all_points = data[['PC1', 'PC2']].values

# 随机采样点对
np.random.seed(42)
random_distances = []

for _ in tqdm(range(n_random_samples)):
    # 随机选择两个不同的点
    idx1, idx2 = np.random.choice(len(all_points), size=2, replace=False)
    distance = np.linalg.norm(all_points[idx1] - all_points[idx2])
    random_distances.append(distance)

random_distances = np.array(random_distances)

print(f"随机距离数量: {len(random_distances)}")
print(f"随机平均距离: {np.mean(random_distances):.4f}")
print(f"随机距离标准差: {np.std(random_distances):.4f}")
print(f"随机距离中位数: {np.median(random_distances):.4f}")


计算随机点对距离（采样 122364 对）...


100%|██████████| 122364/122364 [00:19<00:00, 6351.74it/s]

随机距离数量: 122364
随机平均距离: 0.1527
随机距离标准差: 0.0970
随机距离中位数: 0.1334


In [21]:
from scipy import stats
t_stat, p_value = stats.ttest_ind(intra_class_distances, random_distances)
print(f"\nt-test结果: t-statistic = {t_stat:.4f}, p-value = {p_value:.100e}")

ks_stat, ks_pvalue = stats.ks_2samp(intra_class_distances, random_distances)
print(f"KS-test结果: KS-statistic = {ks_stat:.4f}, p-value = {ks_pvalue:.4e}")



t-test结果: t-statistic = -49.0248, p-value = 0.0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000e+00
KS-test结果: KS-statistic = 0.0792, p-value = 0.0000e+00


In [17]:
fig, ax = plt.subplots(figsize=(3, 3))
distance_df = pd.DataFrame({
    'Distance': np.concatenate([intra_class_distances, random_distances]),
    'Class': ['in'] * len(intra_class_distances) + ['random'] * len(random_distances)
})

sns.violinplot(data=distance_df, x='Class', y='Distance', 
               palette=['#5E9FD1', '#ED7B85'], 
               inner=None, 
               linecolor='black',
               ax=ax)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('distance_comparison.pdf', dpi=300, bbox_inches='tight')
plt.close()